# Objective

This notebook builds baseline models using a small synthetic feature set. It contains a rule-based baseline, a Logistic Regression baseline, and a Random Forest baseline.

The notebook also establishes a baseline feature window and a comparison story that should be read as a precursor to the later leaky and safe experiment notebooks. It documents the model families without introducing extra future-derived columns.


In [ ]:
# Baseline 2: Logistic Regression

Logistic Regression is selected because it provides a simple interpretable linear baseline.

The notebook compares the logistic baseline to the same safe feature matrix used in the leakage investigation. It should explain why the time-aware split is important and why leakage columns are excluded from the honest pipeline.


# Baseline 2: Logistic Regression

Logistic Regression is selected because it provides a simple interpretable linear baseline.

The notebook compares the logistic baseline to the same safe feature matrix used in the leakage investigation. It should explain why the time-aware split is important and why leakage columns are excluded from the honest pipeline.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

features = [
    'content_type', 'category', 'word_count', 'content_age_days', 'author_type', 'has_schema',
    'internal_link_count', 'external_link_count', 'organic_clicks', 'organic_impressions',
    'ctr', 'average_position', 'keyword_count', 'ranking_keywords', 'backlinks',
    'domain_authority', 'sessions', 'bounce_rate', 'avg_session_duration', 'conversions',
    'clicks_7d', 'clicks_30d', 'impressions_7d', 'impressions_30d', 'position_7d', 'position_30d'
]
X = df[features]
y = df['is_declining_label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

num_cols = X.select_dtypes(include='number').columns
cat_cols = [c for c in X.columns if c not in num_cols]
pre = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), num_cols),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('enc', OneHotEncoder(handle_unknown='ignore'))]), cat_cols),
])
model = Pipeline([('prep', pre), ('clf', LogisticRegression(max_iter=500, random_state=42))])
model.fit(X_train, y_train)
print('Logistic Regression baseline trained')
print(f"Logistic train rows: {len(X_train)}; feature columns: {len(features)}")


# Baseline 3: Random Forest

Random Forest captures nonlinear relationships and provides a feature importance table.

This model is deliberately kept close to the same synthetic feature family as the logistic baseline to make the baseline comparison easier to understand.


In [ ]:
# Results

The goal of this notebook is to understand the baseline behavior before the leakage investigation.

By comparing a simple logistic model with a tree-based baseline, the notebook creates a clean baseline story and shows that the same time-aware split and feature window can be evaluated in more than one model family.


# Results

The goal of this notebook is to understand the baseline behavior before the leakage investigation.

The generated model comparison artifact records leaky and safe logistic regression and random forest experiments in `outputs/model_comparison.csv`. In a real portfolio explanation, this table would be read side by side with `experiments/results.csv` to interpret the difference between synthetic baseline performance and the leakage-aware evaluation story.
